In [1]:
!pip install jsonlines
!pip install torch torchvision torchaudio
!pip install datasets
!pip install transformers
!pip install peft
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 58.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

### Preparing Data


In [4]:
dataset = datasets.load_dataset("json", data_files="https://matches.tiiny.site/training.jsonl")["train"]
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['prompt', 'summary'],
    num_rows: 636
})


In [6]:
print(dataset[0]['prompt'])
print(dataset[0]['summary'])

Match Details:
Season: 2017
City: Hyderabad
Venue: Rajiv Gandhi International Stadium, Uppal
Date: 2017-04-05
Home Team: Sunrisers Hyderabad
Away Team: Royal Challengers Bangalore
Based on the details above, provide a concise summary of the match.
Summary: In the match, Sunrisers Hyderabad emerged victorious. The key performer was Yuvraj Singh. The toss was won by Royal Challengers Bangalore, who chose to field. Overall, the match between Sunrisers Hyderabad and Royal Challengers Bangalore was held at Rajiv Gandhi International Stadium, Uppal in Hyderabad on the date 2017-04-05.


In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [10]:
def generate_summary(input, model):
  input_prompt = f"""
                Following are the IPL match details.

                {input}

                Summary :
                """
  input_ids = tokenizer(input_prompt, return_tensors='pt')
  tokenized_output = model.generate(input_ids['input_ids'], min_length=30, max_length=200)
  output = tokenizer.decode(tokenized_output[0], skip_special_tokens=True)
  return output

In [13]:
print(dataset[0]['prompt'])
print('\n')
print(generate_summary(dataset[0]['prompt'], model))

Match Details:
Season: 2017
City: Hyderabad
Venue: Rajiv Gandhi International Stadium, Uppal
Date: 2017-04-05
Home Team: Sunrisers Hyderabad
Away Team: Royal Challengers Bangalore
Based on the details above, provide a concise summary of the match.


The 2017 IPL season starts on April 5th, 2017. The match will be played at the Rajiv Gandhi International Stadium, Uppal.


In [14]:
dataset[0]['summary']

'Summary: In the match, Sunrisers Hyderabad emerged victorious. The key performer was Yuvraj Singh. The toss was won by Royal Challengers Bangalore, who chose to field. Overall, the match between Sunrisers Hyderabad and Royal Challengers Bangalore was held at Rajiv Gandhi International Stadium, Uppal in Hyderabad on the date 2017-04-05.'

# **Finetuning**


In [15]:
def tokenize_inputs(example):
  start_prompt = "Summarize the following IPL Match details. \n\n"
  end_prompt = "\n\nSummary: "
  prompt = [start_prompt + i + end_prompt for i in example["prompt"]]
  example['input_ids'] = tokenizer(prompt, padding="max_length", truncation=True, return_tensors="pt").input_ids
  example['labels'] = tokenizer(example["summary"], padding="max_length", truncation=True, return_tensors="pt").input_ids
  return example

In [16]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer_datasets = dataset.map(tokenize_inputs, batched=True)

Map:   0%|          | 0/636 [00:00<?, ? examples/s]

In [17]:
tokenizer_datasets

Dataset({
    features: ['prompt', 'summary', 'input_ids', 'labels'],
    num_rows: 636
})

In [18]:
tokenizer_datasets = tokenizer_datasets.remove_columns(['prompt', 'summary'])

In [22]:
tokenizer_datasets[0].keys()

dict_keys(['input_ids', 'labels'])

In [23]:
from huggingface_hub import notebook_login
notebook_login()

In [28]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="./flan-t5-base-finetuned",
    hub_model_id="karthikaj6/flan-t5-base-ipl-finetuned",
    learning_rate=2e-5,
    num_train_epochs=5,
    weight_decay=0.01,
    auto_find_batch_size=True,
    logging_steps=10,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenizer_datasets,
    data_collator=data_collator
)

In [29]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
10,1.605900
20,0.496100
30,0.367600
40,0.268700
50,0.212000
60,0.159000
70,0.125300
80,0.109900
90,0.084700
100,0.073800


TrainOutput(global_step=795, training_loss=0.0818276954896795, metrics={'train_runtime': 1096.5705, 'train_samples_per_second': 2.9, 'train_steps_per_second': 0.725, 'total_flos': 2177528380784640.0, 'train_loss': 0.0818276954896795, 'epoch': 5.0})

In [31]:
print(dataset[0]['prompt'])
print("\n")
print(dataset[0]['summary'])

Match Details:
Season: 2017
City: Hyderabad
Venue: Rajiv Gandhi International Stadium, Uppal
Date: 2017-04-05
Home Team: Sunrisers Hyderabad
Away Team: Royal Challengers Bangalore
Based on the details above, provide a concise summary of the match.


Summary: In the match, Sunrisers Hyderabad emerged victorious. The key performer was Yuvraj Singh. The toss was won by Royal Challengers Bangalore, who chose to field. Overall, the match between Sunrisers Hyderabad and Royal Challengers Bangalore was held at Rajiv Gandhi International Stadium, Uppal in Hyderabad on the date 2017-04-05.


In [38]:
import torch

model.to('cuda')

input_prompt = f"""
                Following are the IPL match details.

                {dataset[0]['prompt']}

                Summary :
                """

input_ids = tokenizer(input_prompt, return_tensors='pt').to('cuda')
tokenized_output = model.generate(input_ids['input_ids'], min_length=30, max_length=200)
output = tokenizer.decode(tokenized_output[0], skip_special_tokens=True)
print(output)

The match between Sunrisers Hyderabad and Royal Challengers Bangalore was held at Rajiv Gandhi International Stadium, Uppal in Hyderabad on the date 2017-04-05. The total number of players in the match was 165. The toss was won by Sunrisers Hyderabad, who were bowled out by Royal Challengers Bangalore. The total fielding force was 153.


In [39]:
trainer.push_to_hub()

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

training_args.bin:   0%|          | 0.00/5.37k [00:00<?, ?B/s]

events.out.tfevents.1739377699.9719bcc5dcf7.772.0:   0%|          | 0.00/5.99k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/karthikaj6/flan-t5-base-ipl-finetuned/commit/aa9323766c7c033e5435bf84f18c99fb9289c4c7', commit_message='End of training', commit_description='', oid='aa9323766c7c033e5435bf84f18c99fb9289c4c7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/karthikaj6/flan-t5-base-ipl-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='karthikaj6/flan-t5-base-ipl-finetuned'), pr_revision=None, pr_num=None)

# **PeFT - LoRA Finetuning**

In [53]:
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [54]:
tokenizer_datasets

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 636
})

In [55]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=32, # Rank of the matrix used in LoRA
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.SEQ_2_SEQ_LM #FLAN-T5
  )

In [56]:
peft_model = get_peft_model(model, lora_config)

In [72]:
training_args = TrainingArguments(
    output_dir="./flan-t5-base-finetuned-peft",
    hub_model_id="karthikaj6/flan-t5-base-ipl-finetuned-peft",
    learning_rate=2e-5,
    num_train_epochs=10,
    weight_decay=0.005,
    auto_find_batch_size=True,
    logging_steps=20,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

peft_trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenizer_datasets,
    data_collator=data_collator
)

In [73]:
peft_model.print_trainable_parameters()

trainable params: 3,538,944 || all params: 251,116,800 || trainable%: 1.4093


In [ ]:
peft_trainer.train()

Step,Training Loss
20,0.170500
40,0.161700
60,0.156600
80,0.152300
100,0.141500
120,0.137600
140,0.133900
160,0.127300
180,0.123100
200,0.116500


In [60]:
print(dataset[10]['prompt'])
print("\n")
print(dataset[10]['summary'])

Match Details:
Season: 2017
City: Kolkata
Venue: Eden Gardens
Date: 2017-04-13
Home Team: Kings XI Punjab
Away Team: Kolkata Knight Riders
Based on the details above, provide a concise summary of the match.


Summary: In the match, Kolkata Knight Riders emerged victorious. The key performer was SP Narine. The toss was won by Kolkata Knight Riders, who chose to field. Overall, the match between Kings XI Punjab and Kolkata Knight Riders was held at Eden Gardens in Kolkata on the date 2017-04-13.


In [67]:

model.to('cuda')

input_prompt = f"""
                Following are the IPL match details.

                {dataset[10]['prompt']}

                Summary :
                """

input_ids = tokenizer(input_prompt, return_tensors='pt').to('cuda')
tokenized_output = model.generate(input_ids['input_ids'], min_length=30, max_length=200)
output = tokenizer.decode(tokenized_output[0], skip_special_tokens=True)
print(output)

IPL match details: Season: 2017 City: Kolkata Venue: Eden Gardens Date: 2017-04-13 Home Team: Kings XI Punjab Away Team: Kolkata Knight Riders
